# Bounding box detection - Racoon data


## Data files
- images_racoon.rar: contain images of racoons
- train_labels.cv: contains coordinates for bounding box for every image

### Import the necessary libraries

In [0]:
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import LSTM
from keras.callbacks import ModelCheckpoint
from keras.utils import np_utils
from glob import glob
import seaborn as sns
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import os
import cv2

### Change directory

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=email%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocs.test%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.photos.readonly%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fpeopleapi.readonly&response_type=code

Enter your authorization code:
··········
Mounted at /content/drive


### Load the training data from train.csv file

In [0]:
df = pd.read_csv("/content/drive/My Drive/LSTM/train_labels.csv")

### Print the shape of the train dataset

In [5]:
df.shape

(173, 8)

### Declare a variable IMAGE_SIZE = 128 as we will be using MobileNet which will be taking Input shape as 128 * 128 

In [0]:
IMAGE_SIZE = 128
!mkdir "/content/drive/My Drive/LSTM/resized"
resized_folder = "/content/drive/My Drive/LSTM/resized"

### With the help of csv.reader write a for loop which can load the train.csv file and store the path, width, height, x0,y0,x1,y1 in induvidual variables. <br>
1. Create a list variable known as 'path' which has all the path for all the training images
2. Create an array 'coords' which has the resized coordinates of the bounding box for the training images

<u>Note:</u> All the training images should be downsampled to 128 * 128 as it is the input shape of MobileNet (which we will be using for Object detection). Hence the corresponding coordinates of the bounding boxes should be changed to match the image dimension of 128 * 128 

In [23]:

train_folder = "/content/drive/My Drive/LSTM/images"
images_per_class = {}
path = []
coords = []
print(glob(train_folder))


['/content/drive/My Drive/LSTM/images']


In [26]:
for i,j in df.iterrows():
  w = j['width']
  h = j['height']
  x0 = j['xmin']
  y0 = j['ymin']
  x1 = j['xmax']
  y1 = j['ymax']
  class_folder_path = os.path.join(train_folder, j['filename'])
  path.append(class_folder_path)
  x_ratio = IMAGE_SIZE /w;
  y_ratio = IMAGE_SIZE /h;
  x0_new = x0 * x_ratio
  x1_new = x1 * x_ratio
  y0_new = y0 * y_ratio
  y1_new = y1 * y_ratio
  coords.append((x0_new,y0_new,x1_new,y1_new))
  print (class_folder_path, (x0_new,y0_new,x1_new,y1_new))
  

/content/drive/My Drive/LSTM/images/raccoon-17.jpg (46.94980694980695, 39.58762886597938, 82.53281853281854, 77.85567010309278)
/content/drive/My Drive/LSTM/images/raccoon-11.jpg (0.5818181818181818, 0.2962962962962963, 89.4060606060606, 127.7037037037037)
/content/drive/My Drive/LSTM/images/raccoon-63.jpg (15.786666666666667, 34.24, 59.733333333333334, 92.8)
/content/drive/My Drive/LSTM/images/raccoon-63.jpg (48.42666666666667, 29.76, 85.97333333333334, 95.36)
/content/drive/My Drive/LSTM/images/raccoon-60.jpg (27.194139194139193, 22.832432432432434, 92.36630036630036, 87.87027027027028)
/content/drive/My Drive/LSTM/images/raccoon-69.jpg (7.492682926829268, 5.723577235772359, 117.38536585365854, 124.87804878048782)
/content/drive/My Drive/LSTM/images/raccoon-180.jpg (25.386666666666667, 6.72, 78.50666666666667, 127.68)
/content/drive/My Drive/LSTM/images/raccoon-200.jpg (52.47509578544061, 6.632124352331607, 122.11494252873563, 110.09326424870467)
/content/drive/My Drive/LSTM/images/r

In [27]:
print(path)

['/content/drive/My Drive/LSTM/images/raccoon-17.jpg', '/content/drive/My Drive/LSTM/images/raccoon-11.jpg', '/content/drive/My Drive/LSTM/images/raccoon-63.jpg', '/content/drive/My Drive/LSTM/images/raccoon-63.jpg', '/content/drive/My Drive/LSTM/images/raccoon-60.jpg', '/content/drive/My Drive/LSTM/images/raccoon-69.jpg', '/content/drive/My Drive/LSTM/images/raccoon-180.jpg', '/content/drive/My Drive/LSTM/images/raccoon-200.jpg', '/content/drive/My Drive/LSTM/images/raccoon-141.jpg', '/content/drive/My Drive/LSTM/images/raccoon-19.jpg', '/content/drive/My Drive/LSTM/images/raccoon-84.jpg', '/content/drive/My Drive/LSTM/images/raccoon-124.jpg', '/content/drive/My Drive/LSTM/images/raccoon-182.jpg', '/content/drive/My Drive/LSTM/images/raccoon-111.jpg', '/content/drive/My Drive/LSTM/images/raccoon-91.jpg', '/content/drive/My Drive/LSTM/images/raccoon-79.jpg', '/content/drive/My Drive/LSTM/images/raccoon-93.jpg', '/content/drive/My Drive/LSTM/images/raccoon-20.jpg', '/content/drive/My Dr

### Write a for loop which can load all the training images into a variable 'batch_images' using the paths from the 'paths' variable
<u>Note:</u> Convert the image to RGB scale as the MobileNet accepts 3 channels as inputs   

In [57]:
batch_imgs ={}
batch_imgs['a']=[]
def divide_chunks(l, n): 
      
    # looping till length l 
    print(len(l))
    for i in range(0, len(l), n):
      for j in range(n):
        if(i+j < len(l)):
          image_bgr = cv2.imread(l[i+j], cv2.IMREAD_COLOR)
          screen = cv2.cvtColor(image_bgr, cv2.COLOR_RGB2BGR)
          batch_imgs['a'].append(screen)
          
divide_chunks(path, 10)

        

346


### Import MobileNet and load MobileNet into a variable named 'model' which takes input shape of 128 * 128 * 3. Freeze all the layers. Add convolution and reshape layers at the end to ensure the output is 4 coordinates

In [85]:
import keras
import tensorflow as tf
from keras.applications import MobileNet
mobile = keras.applications.mobilenet.MobileNet()
mobile.summary()

Model: "mobilenet_1.00_224"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         (None, 224, 224, 3)       0         
_________________________________________________________________
conv1_pad (ZeroPadding2D)    (None, 225, 225, 3)       0         
_________________________________________________________________
conv1 (Conv2D)               (None, 112, 112, 32)      864       
_________________________________________________________________
conv1_bn (BatchNormalization (None, 112, 112, 32)      128       
_________________________________________________________________
conv1_relu (ReLU)            (None, 112, 112, 32)      0         
_________________________________________________________________
conv_dw_1 (DepthwiseConv2D)  (None, 112, 112, 32)      288       
_________________________________________________________________
conv_dw_1_bn (BatchNormaliza (None, 112, 112, 32

In [86]:
#mobile.summary()
print(len(mobile.layers))
for layer in mobile.layers[:]:
    layer.trainable = False

93


In [0]:
mobile.layers.pop()
mobile.layers.pop()
mobile.layers.pop()
mobile.layers.pop()
mobile.layers.pop()
mobile.layers.append(tf.keras.layers.Dense(4, activation='linear'))



### Define a custom loss function IoU which calculates Intersection Over Union

In [0]:
def loss(gt,pred):
    intersections = 0
    unions = 0
    diff_width = np.minimum(gt[:,0] + gt[:,2], pred[:,0] + pred[:,2]) - np.maximum(gt[:,0], pred[:,0])
    diff_height = np.minimum(gt[:,1] + gt[:,3], pred[:,1] + pred[:,3]) - np.maximum(gt[:,1], pred[:,1])
    intersection = diff_width * diff_height
    
    # Compute union
    area_gt = gt[:,2] * gt[:,3]
    area_pred = pred[:,2] * pred[:,3]
    union = area_gt + area_pred - intersection

#     Compute intersection and union over multiple boxes
    for j, _ in enumerate(union):
        if union[j] > 0 and intersection[j] > 0 and union[j] >= intersection[j]:
            intersections += intersection[j]
            unions += union[j]

    # Compute IOU. Use epsilon to prevent division by zero
    iou = np.round(intersections / (unions + epsilon()), 4)
    iou = iou.astype(np.float32)
    return iou

def IoU(y_true, y_pred):
    iou = tf.py_func(loss, [y_true, y_pred], tf.float32)
    return iou

### Write model.compile function & model.fit function with: <br>
1. Optimizer = Adam, Loss = 'mse' and metrics = IoU
2. Epochs = 30, batch_size = 32, verbose = 1

In [91]:
from keras.models import Sequential, Model
from keras.layers import Reshape, Activation, Conv2D, Input, MaxPooling2D, BatchNormalization, Flatten, Dense, Lambda
from keras.layers.advanced_activations import LeakyReLU
from keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard
from keras.optimizers import SGD, Adam, RMSprop
from keras.layers.merge import concatenate
import matplotlib.pyplot as plt
import keras.backend as K

mobile.compile(loss='mse',metrics=[IoU],optimizer='Adam')
mobile.summary()


Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to convert a tf eager tensor to
    an ndarray (just call tensor.numpy()) but having access to eager tensors
    means `tf.py_function`s can use accelerators such as GPUs as well as
    being differentiable using a gradient tape.
    - tf.numpy_function maintains the semantics of the deprecated tf.py_func
    (it is not differentiable, and manipulates numpy arrays). It drops the
    stateful argument making all functions stateful.
    
Model: "mobilenet_1.00_224"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         (None, 224, 224, 3)       0         
_________________________________________________________________
conv1_pad (ZeroPadding2D)    (None,

ValueError: ignored

### Pick a test image from the given data

In [0]:
mobile.fit(x_train, y_train,
              batch_size=32, nb_epoch=30,
              verbose=1,
              validation_data=(xmin,ymin,xmax,ymax\))


### Resize the image to 128 * 128 and preprocess the image for the MobileNet model

### Predict the coordinates of the bounding box for the given test image

### Plot the test image using .imshow and draw a boundary box around the image with the coordinates obtained from the model

In [92]:
x0 = int(region[0] * image_width / IMAGE_SIZE) # Scale the BBox
y0 = int(region[1] * image_height / IMAGE_SIZE)

x1 = int((region[2]) * image_width / IMAGE_SIZE)
y1 = int((region[3]) * image_height / IMAGE_SIZE)


import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np


# Create figure and axes
fig,ax = plt.subplots(1)

# Display the image
ax.imshow(unscaled)

# Create a Rectangle patch
rect = patches.Rectangle((x0, y0), (x1 - x0) , (y1 - y0) , linewidth=2, edgecolor='r', facecolor='none')

# Add the patch to the Axes
ax.add_patch(rect)

plt.show()

NameError: ignored